In [2]:
!pip install transformers torch

In [3]:
from transformers import BertTokenizer, BertForSequenceClassification

In [4]:
model_name = "bert-base-uncased"

In [ ]:
# Local file system - no need to mount drive
print("Running on local environment")

Mounted at /content/drive


In [ ]:
import pandas as pd
import os

file_path = os.path.join("..", "data", "cleaned_anxiety_dataset.csv")

df = pd.read_csv(file_path)

print(f"Loaded data with shape: {df.shape}")
df.head()

,statement,label
0,I feel extremely nervous about tomorrow's exam,1
1,My heart races whenever I think about results,1
2,I cannot concentrate because of exam pressure,1
3,I feel calm and prepared for the test,0
4,I studied well and feel confident,0


In [7]:
!pip install transformers torch

In [8]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.model_selection import train_test_split
from torch.optim import AdamW

In [ ]:
file_path = os.path.join("..", "data", "cleaned_anxiety_dataset.csv")
df = pd.read_csv(file_path)

print(f"Dataset shape: {df.shape}")
df.head()

,statement,label
0,I feel extremely nervous about tomorrow's exam,1
1,My heart races whenever I think about results,1
2,I cannot concentrate because of exam pressure,1
3,I feel calm and prepared for the test,0
4,I studied well and feel confident,0


In [10]:
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df["statement"],
    df["label"],
    test_size=0.2,
    random_state=42
)

In [11]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [12]:
class AnxietyDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            texts.tolist(),
            truncation=True,
            padding=True,
            max_length=128
        )
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [13]:
train_dataset = AnxietyDataset(train_texts, train_labels)
val_dataset = AnxietyDataset(val_texts, val_labels)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=3
)

model.to(device)
optimizer = AdamW(model.parameters(), lr=2e-5)
model.train()

for epoch in range(2):
    print(f"\nEpoch {epoch+1}")
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()

    print("Average Loss:", total_loss / len(train_loader))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch 1
Average Loss: 1.0613153159618378

Epoch 2
Average Loss: 0.9238387882709503


In [ ]:
model.eval()

correct = 0
total = 0

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        predictions = torch.argmax(outputs.logits, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total
print("Validation Accuracy:", accuracy)

# Save model to local directory
model_path = os.path.join("..", "model", "anxiety_model")
os.makedirs(model_path, exist_ok=True)
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)
print(f"Model saved to {model_path}")

Validation Accuracy: 0.2857142857142857


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  adding: anxiety_model/ (stored 0%)
  adding: anxiety_model/model.safetensors (deflated 7%)
  adding: anxiety_model/config.json (deflated 54%)
  adding: anxiety_model/tokenizer.json (deflated 71%)
  adding: anxiety_model/tokenizer_config.json (deflated 42%)


In [ ]:
import shutil

zip_path = os.path.join("..", "model", "anxiety_model.zip")
shutil.make_archive(zip_path.replace('.zip', ''), 'zip', os.path.join("..", "model"), "anxiety_model")
print(f"Model packaged to {zip_path}")

updating: anxiety_model/ (stored 0%)
updating: anxiety_model/model.safetensors (deflated 7%)
updating: anxiety_model/config.json (deflated 54%)
updating: anxiety_model/tokenizer.json (deflated 71%)
updating: anxiety_model/tokenizer_config.json (deflated 42%)
